In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

with open("bb84_results.json", "r") as f:
    bb84_results = json.load(f)

with open("lm05_results.json", "r") as f:
    lm05_results = json.load(f)

print(f"BB84 records: {len(bb84_results)}")
print(f"LM05 records: {len(lm05_results)}")

In [ ]:
df_bb84 = pd.DataFrame(bb84_results)
df_lm05 = pd.DataFrame(lm05_results)

df_bb84["protocol"] = "BB84"
df_lm05["protocol"] = "LM05"

df_all = pd.concat([df_bb84, df_lm05], ignore_index=True)
df_all.sort_values(["protocol", "NT", "distance_km"], inplace=True)
df_all.reset_index(drop=True, inplace=True)
df_all

In [ ]:
MIMO_configs = sorted(df_all["NT"].unique())
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig_qber = go.Figure()

for idx, NT in enumerate(MIMO_configs):
    color = colors[idx % len(colors)]

    sub_bb84 = df_bb84[df_bb84["NT"] == NT].sort_values("distance_km")
    fig_qber.add_trace(go.Scatter(
        x=sub_bb84["distance_km"], y=sub_bb84["QBER"],
        mode="lines+markers", name=f"BB84 NT={NT}",
        line=dict(color=color, dash="solid")
    ))

    sub_lm05 = df_lm05[df_lm05["NT"] == NT].sort_values("distance_km")
    fig_qber.add_trace(go.Scatter(
        x=sub_lm05["distance_km"], y=sub_lm05["QBER"],
        mode="lines+markers", name=f"LM05 NT={NT}",
        line=dict(color=color, dash="dash")
    ))

fig_qber.update_layout(
    title="QBER vs Distance — BB84 (solid) vs LM05 (dashed)",
    xaxis_title="Distance (km)",
    yaxis_title="QBER",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig_qber.show()

In [ ]:
fig_skr = go.Figure()

for idx, NT in enumerate(MIMO_configs):
    color = colors[idx % len(colors)]

    sub_bb84 = df_bb84[df_bb84["NT"] == NT].sort_values("distance_km")
    skr_bb84 = sub_bb84["SKR"].clip(lower=1e-12)  # avoid log(0)/negative on log axis
    fig_skr.add_trace(go.Scatter(
        x=sub_bb84["distance_km"], y=skr_bb84,
        mode="lines+markers", name=f"BB84 NT={NT}",
        line=dict(color=color, dash="solid")
    ))

    sub_lm05 = df_lm05[df_lm05["NT"] == NT].sort_values("distance_km")
    skr_lm05 = sub_lm05["SKR"].clip(lower=1e-12)
    fig_skr.add_trace(go.Scatter(
        x=sub_lm05["distance_km"], y=skr_lm05,
        mode="lines+markers", name=f"LM05 NT={NT}",
        line=dict(color=color, dash="dash")
    ))

fig_skr.update_layout(
    title="SKR vs Distance — BB84 (solid) vs LM05 (dashed)",
    xaxis_title="Distance (km)",
    yaxis_title="SKR (bps, log scale)",
    yaxis_type="log",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig_skr.show()

print("Note: negative SKR values clipped to 1e-12 only for the plot's log axis.")
print("Actual SKR sign is preserved in the tables below.")

In [ ]:
def secure_range(df, NT):
    sub = df[(df["NT"] == NT) & (df["SKR"] > 0)]
    return sub["distance_km"].max() if not sub.empty else 0

rows = []
for NT in MIMO_configs:
    rows.append({
        "NT": NT,
        "BB84_secure_range_km": secure_range(df_bb84, NT),
        "LM05_secure_range_km": secure_range(df_lm05, NT),
    })

df_range = pd.DataFrame(rows)
df_range

In [ ]:
fig_range = go.Figure()
fig_range.add_trace(go.Bar(x=df_range["NT"], y=df_range["BB84_secure_range_km"], name="BB84"))
fig_range.add_trace(go.Bar(x=df_range["NT"], y=df_range["LM05_secure_range_km"], name="LM05"))
fig_range.update_layout(
    title="Secure Range (max distance with SKR > 0) vs NT",
    xaxis_title="NT (MIMO config)",
    yaxis_title="Secure range (km)",
    barmode="group",
    template="plotly_white"
)
fig_range.show()

In [ ]:
summary = pd.merge(
    df_bb84[["NT", "distance_km", "QBER", "SKR"]],
    df_lm05[["NT", "distance_km", "QBER", "SKR"]],
    on=["NT", "distance_km"], suffixes=("_BB84", "_LM05")
).sort_values(["NT", "distance_km"]).reset_index(drop=True)

summary

In [ ]:
summary.to_csv("bb84_vs_lm05_comparison.csv", index=False)
print("Saved -> bb84_vs_lm05_comparison.csv")